```markdown
# Aprendendo Regras de Associação com o Algoritmo Apriori 🛒

Este notebook apresenta de forma prática e detalhada como descobrir padrões ocultos em conjuntos de transações comerciais utilizando o algoritmo **Apriori**. Você aprenderá a identificar quais produtos costumam ser comprados juntos, permitindo criar estratégias de recomendação e organização de produtos.

### Objetivos de Aprendizado:
* Compreender o conceito de Regras de Associação e sua utilidade prática.
* Entender o funcionamento do Algoritmo Apriori para a filtragem de conjuntos de itens frequentes.
* Calcular e interpretar as métricas de **Suporte** e **Confiança**.
* Utilizar a biblioteca `mlxtend` para aplicar o algoritmo sobre dados transacionais reais.
```

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


```markdown
## Passo 1: Importação das Bibliotecas Necessárias

No bloco acima, importamos as ferramentas essenciais para o nosso estudo:
* **`pandas` (`pd`)**: A biblioteca padrão para manipulação de dados estruturados em formato de tabelas (DataFrames).
* **`TransactionEncoder`**: Uma ferramenta de pré-processamento da biblioteca `mlxtend` que transforma listas de compras textuais em uma matriz binária (verdadeiro/falso), formato exigido pelo algoritmo Apriori.
* **`apriori`**: O algoritmo que usaremos para identificar quais grupos de itens aparecem juntos com frequência.
* **`association_rules`**: A função que gera as regras matemáticas de associação (ex: "Quem compra Pão, também compra Manteiga") a partir dos itens frequentes gerados pelo Apriori.
```

In [ ]:
with open('Transacoes.txt',"r") as f:
  transactions = [line.strip().split(",") for line in f.readlines()]

transactions

```markdown
## Passo 2: Carregamento dos Dados de Transações

### O que este passo faz:
Lê o arquivo texto `Transacoes.txt` linha por linha, separando os produtos de cada compra (transação) por vírgula e removendo espaços em branco extras.

### Por que ele é necessário?
Antes de aplicar qualquer modelo matemático, precisamos carregar os nossos dados brutos para a memória. O resultado é uma lista de listas (uma estrutura bidimensional), onde cada sublista representa o "carrinho de compras" de um cliente específico.
```

In [ ]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

```markdown
## Passo 3: Pré-processamento com TransactionEncoder

### O que este passo faz e por que é necessário:
O algoritmo Apriori não consegue ler listas de texto diretamente de forma eficiente. Ele exige uma tabela binária conhecida como **Representação One-Hot**.

O `TransactionEncoder` mapeia todos os itens únicos existentes em todas as compras e cria uma coluna para cada um deles. Se o cliente comprou aquele item específico na transação, a célula correspondente recebe `True`; caso contrário, recebe `False`.

💡 **Para pensar**: Por que usamos valores booleanos (`True`/`False`) em vez de strings diretamente no algoritmo? Pense sobre o custo computacional e velocidade de processamento quando lidamos com milhões de compras.
```

In [ ]:
df

```markdown
## Passo 4: Geração de Itemsets Frequentes com Apriori

### Conceito Chave: Suporte
O **Suporte** é a proporção (ou porcentagem) de transações que contêm um determinado conjunto de itens. Se temos 10 transações e um item aparece em 5 delas, seu suporte é de 0.5 (ou 50%).

### O que este passo faz:
Filtra e exibe apenas os conjuntos de itens (itemsets) que possuem um suporte maior ou igual a **0.5** (50%). Isso nos ajuda a ignorar combinações raras de produtos e focar apenas no que é verdadeiramente comum no comportamento do consumidor.
```

In [ ]:
frequent_itemsets = apriori(df, min_support=0.5, use_colnames=True)
frequent_itemsets

```markdown
## Passo 5: Geração das Regras de Associação

### Conceito Chave: Confiança
A **Confiança** mede a força da regra. Ela responde à pergunta: "De todas as pessoas que compraram o item X, qual a porcentagem de pessoas que também compraram o item Y?". Uma confiança de 0.5 significa que há 50% de chance de o cliente levar o segundo produto dado que levou o primeiro.

### O que este passo faz:
Aplica a função `association_rules` sobre os nossos conjuntos frequentes criados anteriormente, exigindo que as regras tenham uma confiança mínima de **0.5** (50%).
```

In [ ]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

In [ ]:
print(rules)

```markdown
## Passo 6: Análise dos Resultados Obtidos

Ao visualizar a tabela final de regras, observe as colunas principais:
* **`antecedents`**: O produto ou conjunto de produtos que o cliente já possui no carrinho (o ponto de partida, "Se comprou...").
* **`consequents`**: O produto que o cliente tende a comprar em seguida ("então também comprará...").
* **`support`**: A presença geral dessa combinação em todo o banco de dados.
* **`confidence`**: A precisão da regra ou taxa de acerto esperado.
* **`lift`**: Mede o quão mais provável é a compra do consequente dado o antecedente, comparado à compra do consequente de forma totalmente aleatória. Um `lift > 1` indica uma associação positiva forte!

---

## 🛠️ Erros Comuns e Dicas de Resolução
1. **Erro de arquivo não encontrado (`FileNotFoundError`)**: Se ocorrer ao abrir o arquivo `Transacoes.txt`, certifique-se de que fez o upload do arquivo para a área de arquivos temporários do Colab antes de executar a célula.
2. **Tabela de Regras Vazia**: Se você definir o `min_support` ou `min_threshold` (confiança) muito altos (ex: `0.95`), o algoritmo pode não encontrar nenhuma combinação que atenda a critérios tão rígidos. Reduza os limites gradualmente.
3. **Uso excessivo de memória**: Em bases de dados reais e gigantes, usar limites de suporte muito baixos (ex: `0.01`) pode gerar bilhões de combinações, travando o ambiente de execução.

---

## 🎓 Resumo do Aprendizado
* **Regras de Associação** nos ajudam a descobrir correlações comerciais em bases de transações.
* O **Apriori** otimiza essa busca eliminando previamente os conjuntos de itens que não atingem um suporte mínimo.
* O **Suporte** mede a frequência de um evento, enquanto a **Confiança** mede a certeza de uma consequência lógica.

---

## 📝 Exercícios Propostos

1. **[Fácil]** Altere o valor de `min_support` no algoritmo Apriori para `0.3` e observe se surgem novos itens frequentes na tabela. Explique o porquê da mudança ocorrida.
2. **[Médio]** Modifique a geração de regras de associação para usar a métrica de `lift` com um threshold de `1.2` em vez de usar `confidence` direta. Quais regras foram descartadas?
3. **[Desafiador]** Escreva um código Python simples para contar quantas transações únicas existem em seu arquivo de transações e determine qual é o produto isolado mais vendido de todos sem utilizar a biblioteca `mlxtend`.

---

## 🚀 Para ir além (Tópicos para Pesquisa)
* **Algoritmo FP-Growth**: Uma alternativa muito mais rápida e moderna ao Apriori que não exige gerar candidatos intermediários.
* **Análise de Cesta de Compras (Market Basket Analysis)**: Como grandes varejistas utilizam esses algoritmos para organizar gôndolas físicas de supermercados e sistemas de recomendação online.
```